# Blog Post Summarizer
### Using Hugging Face Transformers to summarize long-form blog content

This notebook demonstrates how to use a pre-trained NLP model to automatically 
generate structured summaries of blog posts.

**Model:** `facebook/bart-large-cnn`  
**Task:** Abstractive text summarization  
**Author:** Your Name  
**Date:** April 2026

---

## The Problem

Blog posts are long. Readers often want to know if a post is worth reading 
before committing. A good automatic summarizer can:
- Generate a TL;DR for each post
- Help readers navigate long-form content
- Demonstrate practical NLP in a real-world context

## The Approach

We use Facebook's BART model with a paragraph-based chunking strategy to 
handle posts that exceed the model's token limit. Along the way we'll 
demonstrate why naive truncation fails and how chunking solves it.

## Setup: Imports and Model Loading

In [1]:
# Core libraries
from transformers import pipeline
import textwrap
import os

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
print("Loading model... (this may take a moment if not cached)")

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

print("Model loaded successfully!")

Loading model... (this may take a moment if not cached)


/opt/conda/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successfully!


## Step 1: Load a Blog Post

We'll load a blog post as a string and examine some basic statistics before 
passing it to the model. This gives us a clear picture of what we're working 
with — particularly word count, which determines how we need to handle the 
model's token limit.

In [3]:
def load_post_from_file(filename):
    """
    Load a blog post from a text file in the notebooks directory.
    
    Args:
        filename: Name of the .txt file (e.g. 'my-post.txt')
    
    Returns:
        Post text as a string
    """
    filepath = os.path.join(os.getcwd(), filename)
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    return text

# Load the post — swap in your own filename here
post = load_post_from_file('blog_engineers.txt')

# Display basic stats
word_count = len(post.split())
print(f"Post loaded successfully")
print(f"Word count:      {word_count}")
print(f"Character count: {len(post)}")
print(f"\nFirst 200 characters preview:")
print("-" * 40)
print(post[:200])

Post loaded successfully
Word count:      1331
Character count: 7895

First 200 characters preview:
----------------------------------------
There are gaggles of geese.

A group of bears is referred to as a sloth of bears.

There are the well-known teams of oxen, hives of bees, and packs of wolves.

We even collectively group people togeth


## Step 2: Handling the Token Limit

BART has a maximum input length of 1024 tokens (roughly 750-800 words). 
Our post exceeds that limit, so we can't feed it to the model directly.

### The Naive Approach — Truncation
The simplest solution is to just cut the post off at the token limit. 
However this has a critical flaw: blog posts don't follow the "inverted 
pyramid" structure that news articles do. In a news article the most 
important information is always first. In a blog post the key content 
might be anywhere.

To demonstrate this, here's what truncation produces on our post:

In [4]:
# Truncate to first 3000 characters and summarize
truncated = post[:3000]

truncation_summary = summarizer(
    truncated,
    max_length=150,
    min_length=50,
    do_sample=False
)

print("=== TRUNCATION APPROACH ===")
print(f"Input: first {len(truncated)} characters of {len(post)} total\n")
print("Result:")
print(textwrap.fill(truncation_summary[0]['summary_text'], width=60))
print("\nProblem: The model only saw the introduction, not the")
print("actual content of the post. The summary is misleading.")

=== TRUNCATION APPROACH ===
Input: first 3000 characters of 7895 total

Result:
There is shockingly little in the way of generally agreed
upon group nomenclature for some of the finest groups of
humans to ever walk this earth. The following is an attempt
to categorize and collectively identify these amazing groups
of smart and talented people.

Problem: The model only saw the introduction, not the
actual content of the post. The summary is misleading.


## Step 3: The Better Approach — Paragraph-Based Chunking

Rather than truncating, we split the post into chunks that respect natural 
paragraph boundaries. This ensures:

1. Every section of the post is represented in the final summary
2. The model never sees incomplete sentences or broken thoughts
3. The content structure of the original post is preserved

This approach is known as **map-reduce summarization**:
- **Map:** summarize each chunk independently
- **Reduce:** combine the results into a final output

In [5]:
def chunk_by_paragraph(text, max_chunk_words=150):
    """
    Split text into chunks that respect paragraph boundaries.
    
    Rather than cutting at a fixed character count, we accumulate 
    complete paragraphs until we approach the word limit, then 
    start a new chunk. This prevents mid-sentence cuts that 
    confuse the model.
    
    Args:
        text: Full blog post text
        max_chunk_words: Max words per chunk before starting a new one
    
    Returns:
        List of text chunks, each containing complete paragraphs
    """
    # Split on double newlines (paragraph breaks)
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for paragraph in paragraphs:
        paragraph_words = len(paragraph.split())
        
        # If adding this paragraph exceeds limit, save current 
        # chunk and start a new one
        if current_word_count + paragraph_words > max_chunk_words \
                and current_chunk:
            chunks.append('\n\n'.join(current_chunk))
            current_chunk = [paragraph]
            current_word_count = paragraph_words
        else:
            current_chunk.append(paragraph)
            current_word_count += paragraph_words
    
    # Capture the final chunk
    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))
    
    return chunks


# Chunk the post and show the structure
chunks = chunk_by_paragraph(post, max_chunk_words=150)

print(f"Post split into {len(chunks)} chunks")
print(f"{'Chunk':<10} {'Words':>6}")
print("-" * 20)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1:<4}  {len(chunk.split()):>6}")

Post split into 10 chunks
Chunk       Words
--------------------
Chunk 1        137
Chunk 2        144
Chunk 3        141
Chunk 4        136
Chunk 5        108
Chunk 6        133
Chunk 7        144
Chunk 8        124
Chunk 9        148
Chunk 10       116


## Step 4: Summarizing Each Section

With clean paragraph-based chunks in place, we run each section through 
BART independently and present the results as a structured summary report.

Rather than forcing a second compression pass — which we found loses too 
much context and nuance — we present section-by-section summaries that 
preserve the flow of the original post.

In [6]:
def summarize_blog_post(text, summarizer, max_chunk_words=150):
    """
    Summarize a blog post using paragraph-based chunking.
    
    Args:
        text: Full blog post text
        summarizer: Hugging Face summarization pipeline
        max_chunk_words: Max words per chunk (default 150)
    
    Returns:
        dict containing chunk_summaries and post stats
    """
    stats = {
        'word_count': len(text.split()),
        'char_count': len(text),
        'chunk_count': 0
    }
    
    chunks = chunk_by_paragraph(text, max_chunk_words)
    stats['chunk_count'] = len(chunks)
    
    print(f"Summarizing {len(chunks)} sections...")
    summaries = []
    
    for i, chunk in enumerate(chunks):
        print(f"  Section {i+1} of {len(chunks)}...")
        
        if len(chunk.split()) < 30:
            continue
        
        summary = summarizer(
            chunk,
            max_length=60,
            min_length=20,
            do_sample=False
        )
        
        summaries.append(summary[0]['summary_text'])
    
    return {
        'chunk_summaries': summaries,
        'stats': stats
    }


# Run the summarizer
result = summarize_blog_post(post, summarizer)

# Display the report
print("\n")
print("=" * 60)
print("BLOG POST SUMMARY REPORT")
print("=" * 60)
print(f"Word count:  {result['stats']['word_count']}")
print(f"Sections:    {result['stats']['chunk_count']}")
print("=" * 60)

for i, summary in enumerate(result['chunk_summaries']):
    print(f"\nSection {i+1}:")
    print(textwrap.fill(summary, width=60, initial_indent="  "))

print("\n" + "=" * 60)

Summarizing 10 sections...
  Section 1 of 10...
  Section 2 of 10...
  Section 3 of 10...
  Section 4 of 10...
  Section 5 of 10...
  Section 6 of 10...
  Section 7 of 10...
  Section 8 of 10...
  Section 9 of 10...
  Section 10 of 10...


BLOG POST SUMMARY REPORT
Word count:  1331
Sections:    10

Section 1:
  There is shockingly little in the way of generally agreed
upon group nomenclature for some of the finest groups of
humans to ever walk this earth.

Section 2:
  Three physicists and three Engineers found themselves on a
train platform heading to a big conference. The physicists
watched in confusion as only one Engineer went to the ticket
window and purchased a ticket.

Section 3:
  The following is an attempt to categorize and collectively
identify these amazing groups of smart and talented people.
Dealing with molecules, materials, and moles, chemical
engineers convert raw materials into usable stuff.

Section 4:
   chemical engineers are collectively known as chemical
engineer

## Conclusions

### What We Built
A blog post summarizer using Facebook's BART model with paragraph-based 
chunking that produces structured, section-by-section summaries of 
long-form content.

### Key Findings

**Truncation fails for blog content.**
BART was fine-tuned on CNN/DailyMail news articles which follow an inverted 
pyramid structure — most important information first. Blog posts don't follow 
this pattern. Naive truncation only summarizes the introduction, producing 
misleading results.

**Paragraph-based chunking outperforms character-based chunking.**
Splitting on paragraph boundaries preserves complete thoughts and prevents 
mid-sentence cuts that confuse the model. Every section of the post is 
represented in the output.

**Section-by-section output outperforms forced compression.**
A second summarization pass (summarizing the summaries) loses too much 
context. Presenting individual section summaries preserves more meaning 
and produces more useful output for a reader.

### Limitations
- BART loses the voice and humor of creative writing — it was trained on 
  factual news content
- Inference is slow on CPU (~5 seconds per chunk)
- Section summaries are useful but not a true single-paragraph abstractive 
  summary of the whole post

### Next Steps
- Test alternative models (PEGASUS, T5) on the same content
- Fine-tune BART on a custom blog post dataset for better style matching
- Build a Gradio UI so the summarizer is usable without the notebook
- Automate TL;DR injection at the top of each published blog post

## Appendix: Using This Notebook on Your Own Posts

1. Save your blog post as a `.txt` file in the `notebooks/` folder
2. Update the filename in the cell below
3. Run all cells from top to bottom (Kernel → Restart Kernel and Run All Cells)

In [7]:
# To summarize a different post, update the filename here and re-run
# post = load_post_from_file('your-post.txt')
# result = summarize_blog_post(post, summarizer)